In [0]:
-- ============================================================
-- Notebook 02: Delta Lake Introduction
-- Author: Alejandra Kheng
-- Description: Demonstrates Delta Lake fundamentals including
--              creating Delta tables, time travel, and MERGE INTO
--              upsert logic used in production data pipelines.
-- ============================================================

-- Use our existing database
USE healthcare_analytics;

-- Create a Delta table for project snapshots
CREATE TABLE IF NOT EXISTS healthcare_analytics.project_snapshots
USING DELTA AS
SELECT
    project_id,
    project_name,
    department_id,
    completed_tasks,
    total_tasks,
    ROUND(
        completed_tasks / NULLIF(total_tasks, 0) * 100, 1
    ) AS completion_pct,
    CURRENT_TIMESTAMP() AS snapshot_date
FROM healthcare_analytics.projects;

-- Verify the table was created
SELECT * FROM healthcare_analytics.project_snapshots
ORDER BY project_id;

project_id,project_name,department_id,completed_tasks,total_tasks,completion_pct,snapshot_date
101,EHR Migration,1,45,50,90.0,2026-06-01T20:12:14.274Z
102,Patient Intake Redesign,1,20,50,40.0,2026-06-01T20:12:14.274Z
103,BI Dashboard Rollout,2,48,50,96.0,2026-06-01T20:12:14.274Z
104,Claims Data Cleanup,2,30,50,60.0,2026-06-01T20:12:14.274Z
105,Audit Prep Q1,3,50,50,100.0,2026-06-01T20:12:14.274Z
106,Compliance Review,3,35,50,70.0,2026-06-01T20:12:14.274Z
107,Data Quality Framework,4,42,50,84.0,2026-06-01T20:12:14.274Z
108,Report Automation,4,15,50,30.0,2026-06-01T20:12:14.274Z


In [0]:
-- ============================================================
-- Simulate a data update: projects have progressed
-- ============================================================

-- Update two projects to show completion progress
UPDATE healthcare_analytics.project_snapshots
SET 
    completed_tasks = 48,
    completion_pct = 96.0,
    snapshot_date = CURRENT_TIMESTAMP()
WHERE project_id = 102; -- Patient Intake Redesign was at 40%

UPDATE healthcare_analytics.project_snapshots
SET 
    completed_tasks = 45,
    completion_pct = 90.0,
    snapshot_date = CURRENT_TIMESTAMP()
WHERE project_id = 108; -- Report Automation was at 30%

-- Verify the updates
SELECT 
    project_id,
    project_name,
    completed_tasks,
    completion_pct,
    snapshot_date
FROM healthcare_analytics.project_snapshots
WHERE project_id IN (102, 108)
ORDER BY project_id;


project_id,project_name,completed_tasks,completion_pct,snapshot_date
102,Patient Intake Redesign,48,96.0,2026-06-01T20:14:19.134Z
108,Report Automation,45,90.0,2026-06-01T20:14:25.726Z


In [0]:
-- ============================================================
-- Delta Lake Time Travel
-- Query previous versions of the data
-- ============================================================

-- See the full history of changes to this table
DESCRIBE HISTORY healthcare_analytics.project_snapshots;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-01T20:14:33.000Z,78631428742542,agabriel0811@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4358172453342664),199ae54e-a0ea-4c64-87ac-300ba7efa95f,0601-201134-12ugua3k-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4574, p25FileSize -> 2516, numDeletionVectorsRemoved -> 1, minFileSize -> 2516, numAddedFiles -> 1, maxFileSize -> 2516, p75FileSize -> 2516, p50FileSize -> 2516, numAddedBytes -> 2516)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-06-01T20:14:31.000Z,78631428742542,agabriel0811@outlook.com,UPDATE,"Map(predicate -> [""(project_id#11733 = 108)""])",null,List(4358172453342664),199ae54e-a0ea-4c64-87ac-300ba7efa95f,0601-201134-12ugua3k-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 3345, conflictDetectionTimeMs -> 603, numDeletionVectorsUpdated -> 1, scanTimeMs -> 1350, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2096, rewriteTimeMs -> 1995)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-06-01T20:14:29.000Z,78631428742542,agabriel0811@outlook.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4358172453342664),74012592-93c4-4ac5-aa46-88cf6587a99e,0601-201134-12ugua3k-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 4543, p25FileSize -> 2478, numDeletionVectorsRemoved -> 1, minFileSize -> 2478, numAddedFiles -> 1, maxFileSize -> 2478, p75FileSize -> 2478, p50FileSize -> 2478, numAddedBytes -> 2478)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-06-01T20:14:25.000Z,78631428742542,agabriel0811@outlook.com,UPDATE,"Map(predicate -> [""(project_id#11337 = 102)""])",null,List(4358172453342664),74012592-93c4-4ac5-aa46-88cf6587a99e,0601-201134-12ugua3k-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 5463, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2869, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2126, rewriteTimeMs -> 2548)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
0,2026-06-01T20:12:26.000Z,78631428742542,agabriel0811@outlook.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4358172453342664),746f978f-bfec-4394-b5ea-88202b40809e,0601-201134-12ugua3k-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 8, numOutputBytes -> 2417)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
-- ============================================================
-- Query a previous version of the table (time travel)
-- ============================================================

-- See what the data looked like at version 0 (original)
SELECT
    project_id,
    project_name,
    completed_tasks,
    completion_pct,
    snapshot_date
FROM healthcare_analytics.project_snapshots VERSION AS OF 0
WHERE project_id IN (102, 108)
ORDER BY project_id;

project_id,project_name,completed_tasks,completion_pct,snapshot_date
102,Patient Intake Redesign,20,40.0,2026-06-01T20:12:14.274Z
108,Report Automation,15,30.0,2026-06-01T20:12:14.274Z


In [0]:
-- ============================================================
-- MERGE INTO: Upsert logic (insert or update)
-- Used in production pipelines to sync source data
-- ============================================================

-- Create a staging table simulating incoming updated data
CREATE OR REPLACE TABLE healthcare_analytics.project_updates AS
SELECT * FROM (
    VALUES
        (102, 'Patient Intake Redesign', 1, 50, 50, 100.0),
        (108, 'Report Automation',       4, 50, 50, 100.0),
        (109, 'New Analytics Platform',  4, 10, 50,  20.0)
) AS updates(
    project_id, project_name, department_id,
    completed_tasks, total_tasks, completion_pct
);

-- MERGE: update existing projects, insert new ones
MERGE INTO healthcare_analytics.project_snapshots AS target
USING healthcare_analytics.project_updates AS source
ON target.project_id = source.project_id

WHEN MATCHED THEN UPDATE SET
    target.completed_tasks = source.completed_tasks,
    target.completion_pct  = source.completion_pct,
    target.snapshot_date   = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    project_id, project_name, department_id,
    completed_tasks, total_tasks, completion_pct, snapshot_date
) VALUES (
    source.project_id, source.project_name, source.department_id,
    source.completed_tasks, source.total_tasks, source.completion_pct,
    CURRENT_TIMESTAMP()
);

-- Verify results: should show 9 rows now with updated values
SELECT
    project_id,
    project_name,
    completed_tasks,
    completion_pct,
    snapshot_date
FROM healthcare_analytics.project_snapshots
ORDER BY project_id;

project_id,project_name,completed_tasks,completion_pct,snapshot_date
101,EHR Migration,45,90.0,2026-06-01T20:12:14.274Z
102,Patient Intake Redesign,50,100.0,2026-06-01T20:17:13.984Z
103,BI Dashboard Rollout,48,96.0,2026-06-01T20:12:14.274Z
104,Claims Data Cleanup,30,60.0,2026-06-01T20:12:14.274Z
105,Audit Prep Q1,50,100.0,2026-06-01T20:12:14.274Z
106,Compliance Review,35,70.0,2026-06-01T20:12:14.274Z
107,Data Quality Framework,42,84.0,2026-06-01T20:12:14.274Z
108,Report Automation,50,100.0,2026-06-01T20:17:13.984Z
109,New Analytics Platform,10,20.0,2026-06-01T20:17:13.984Z
